Extract the dataset to jupyter notebook

In [1]:
import zipfile
with zipfile.ZipFile("ManchesterCSCoordinatedDiabetesStudy-V1.0.3.zip", "r") as zip_ref:
    zip_ref.extractall("minha_pasta")


Im going to work with only 1 person. Here, i create the path to the used files

In [ ]:
mealsdataset = 'minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv'
glucosedataset = 'minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv'
bolusdataset = 'minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv'

created to find the path for the datasets

In [ ]:


import csv

with open('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv', mode='r', encoding='utf-8') as arquivo:
    leitor = csv.reader(arquivo)
    for i, linha in enumerate(leitor):
        if i < 10:
            print(linha)
        else:
            break

In [ ]:
# for each meal event:
for meal in mealsdataset:
    start_time = meal.timestamp
    end_time = start_time + 4h    # esta linha nao funciona
    glucose_window = glucosedataset[(glucosedataset.timestamp >= start_time) & (glucosedataset.timestamp <= end_time)]
# if there is another meal before the 4h
next_meal = get_next_meal(meal)
if next_meal.timestamp < end_time:
    end_time = next_meal.timestamp


find a peak (PPGR_peak)

In [ ]:
peak_value = glucose_window['glucose'].max()
peak_time = glucose_window.loc[glucose_window['glucose'].idxmax(), 'timestamp']


create an event

In [ ]:
{
    "case_id": meal.id,
    "event_type": "PPGR_peak",
    "timestamp": peak_time,
    "value": peak_value
}

In [ ]:
find when glucose comes to normal

In [ ]:
baseline = glucose_before_meal['glucose'].mean()


In [ ]:
find the first value that is below the baseline

In [ ]:
return_time = glucose_window[glucose_window['glucose'] <= baseline].timestamp.min()


In [ ]:
create the event

In [ ]:
join datasets

In [ ]:
events = pd.concat([meals_events, bolus_events, peaks_events, return_events])


In [ ]:
generate the event log

In [ ]:
from pm4py.objects.log.util import dataframe_utils
from pm4py.objects.conversion.log import converter as log_converter

event_log = log_converter.apply(events)


In [ ]:
graph visualization 

In [ ]:
plt.plot(glucose_window['timestamp'], glucose_window['glucose'], '-o', label='Glucose')
plt.axvline(meal.timestamp, color='green', linestyle='--', label='Meal')
plt.scatter(peak_time, peak_value, color='red', marker='^', label='Peak')
plt.axvline(return_time, color='orange', linestyle='--', label='Return to baseline')
plt.legend()
